Dataset Structure:

1. id: unique id for a news article
2. title: title of a news article
3. author: author of a news article
4. text: text of the article
5. label: marks whether an article is real or fake

In [1]:
import numpy as np
import pandas as pd
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [3]:
print(stopwords.words('english'))
# Words that don't add value when analyzing text.

['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been', 'before', 'being', 'below', 'between', 'both', 'but', 'by', 'can', 'couldn', "couldn't", 'd', 'did', 'didn', "didn't", 'do', 'does', 'doesn', "doesn't", 'doing', 'don', "don't", 'down', 'during', 'each', 'few', 'for', 'from', 'further', 'had', 'hadn', "hadn't", 'has', 'hasn', "hasn't", 'have', 'haven', "haven't", 'having', 'he', "he'd", "he'll", 'her', 'here', 'hers', 'herself', "he's", 'him', 'himself', 'his', 'how', 'i', "i'd", 'if', "i'll", "i'm", 'in', 'into', 'is', 'isn', "isn't", 'it', "it'd", "it'll", "it's", 'its', 'itself', "i've", 'just', 'll', 'm', 'ma', 'me', 'mightn', "mightn't", 'more', 'most', 'mustn', "mustn't", 'my', 'myself', 'needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on', 'once', 'only', 'or', 'other', 'our', 'ours', 'ourselves', 'out', 'over', 'own', 're', 's', 'same', 'shan', "shan't", 'she

Data pre-processing

In [5]:
# Load into pandas
news_dataset = pd.read_csv('/content/train.csv')
news_dataset.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


In [6]:
news_dataset.shape

(20800, 5)

In [8]:
# Check for missing values
news_dataset.isnull().sum()

,0
id,0
title,558
author,1957
text,39
label,0


In [9]:
# Replace null values with empty string as dataset is large enough to still train model

news_dataset = news_dataset.fillna('')

In [10]:
# Separate data and labels
X = news_dataset.drop(['label'], axis=1)
Y = news_dataset['label']

In [11]:
print(X)
print(Y)

          id  ...                                               text
0          0  ...  House Dem Aide: We Didn’t Even See Comey’s Let...
1          1  ...  Ever get the feeling your life circles the rou...
2          2  ...  Why the Truth Might Get You Fired October 29, ...
3          3  ...  Videos 15 Civilians Killed In Single US Airstr...
4          4  ...  Print \nAn Iranian woman has been sentenced to...
...      ...  ...                                                ...
20795  20795  ...  Rapper T. I. unloaded on black celebrities who...
20796  20796  ...  When the Green Bay Packers lost to the Washing...
20797  20797  ...  The Macy’s of today grew from the union of sev...
20798  20798  ...  NATO, Russia To Hold Parallel Exercises In Bal...
20799  20799  ...    David Swanson is an author, activist, journa...

[20800 rows x 4 columns]
0        1
1        0
2        1
3        1
4        1
        ..
20795    0
20796    0
20797    0
20798    1
20799    1
Name: label, Length: 2080

*Stemming*: The process of reducing a word to it's root word

In [12]:
port_stem = PorterStemmer()


In [13]:
def stemming(text):
  stemmed_text = re.sub('[^a-zA-Z]',' ', text)
  stemmed_text = stemmed_text.lower()
  stemmed_text = stemmed_text.split()
  stemmed_text = [port_stem.stem(word) for word in stemmed_text if not word in stopwords.words('english')]
  stemmed_text = ' '.join(stemmed_text)
  return stemmed_text

In [14]:
news_dataset['text'] = news_dataset['text'].apply(stemming)

In [15]:
print(news_dataset['text'])

0        hous dem aid even see comey letter jason chaff...
1        ever get feel life circl roundabout rather hea...
2        truth might get fire octob tension intellig an...
3        video civilian kill singl us airstrik identifi...
4        print iranian woman sentenc six year prison ir...
                               ...                        
20795    rapper unload black celebr met donald trump el...
20796    green bay packer lost washington redskin week ...
20797    maci today grew union sever great name america...
20798    nato russia hold parallel exercis balkan press...
20799    david swanson author activist journalist radio...
Name: text, Length: 20800, dtype: object


Convert Text to Vectors

In [17]:
X = news_dataset['text'].values
Y = news_dataset['label'].values

In [18]:
print(X)
print(Y)

['hous dem aid even see comey letter jason chaffetz tweet darrel lucu octob subscrib jason chaffetz stump american fork utah imag courtesi michael jolley avail creativ common licens apolog keith olbermann doubt worst person world week fbi director jame comey accord hous democrat aid look like also know second worst person well turn comey sent infam letter announc fbi look email may relat hillari clinton email server rank democrat relev committe hear comey found via tweet one republican committe chairmen know comey notifi republican chairmen democrat rank member hous intellig judiciari oversight committe agenc review email recent discov order see contain classifi inform long letter went oversight committe chairman jason chaffetz set polit world ablaz tweet fbi dir inform fbi learn exist email appear pertin investig case reopen jason chaffetz jasoninthehous octob cours know case comey actual say review email light unrel case know anthoni weiner sext teenag appar littl thing fact matter c

In [19]:
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(X)

In [20]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5077895 stored elements and shape (20800, 110429)>
  Coords	Values
  (0, 43806)	0.08703559681156282
  (0, 23700)	0.07009533942998004
  (0, 1870)	0.10309872061663368
  (0, 31125)	0.04529680654129223
  (0, 86453)	0.039920963086541125
  (0, 18947)	0.2686115146789949
  (0, 55291)	0.16128879705965116
  (0, 48287)	0.15585196242861912
  (0, 16168)	0.5935000577297513
  (0, 100055)	0.12133995257889343
  (0, 22668)	0.22796177896591885
  (0, 57198)	0.10594943768584071
  (0, 69116)	0.03734273616326195
  (0, 93327)	0.029856833298770362
  (0, 93111)	0.03909774077404906
  (0, 3353)	0.012636792658322085
  (0, 34391)	0.042018817060949336
  (0, 102455)	0.06985897849015361
  (0, 45245)	0.021522001238297356
  (0, 20761)	0.06746822105607313
  (0, 61642)	0.021883245870494866
  (0, 49216)	0.06419718762360735
  (0, 6904)	0.02206073247828088
  (0, 21055)	0.02870620923196509
  (0, 19043)	0.021392573900426527
  :	:
  (20799, 107985)	0.0509215542269435

Training/Test Splitting

In [23]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, stratify = Y, random_state = 2)

Model Training

In [24]:
model = LogisticRegression()

In [25]:
model.fit(X_train, Y_train)

LogisticRegression()

Model Evaluation

In [26]:
# Accuracy on training data

train_prediction = model.predict(X_train)
train_pred_acc = accuracy_score(train_prediction, Y_train)

In [27]:
print(train_pred_acc)

0.9713341346153846


In [28]:
# Accuracy on test data

test_prediction = model.predict(X_test)
test_pred_acc = accuracy_score(test_prediction, Y_test)

In [29]:
print(test_pred_acc)

0.9389423076923077


Prediction System

In [33]:
input_article_text = X_test[1]

prediction = model.predict(input_article_text)
print(input_article_text)
print(prediction)

if prediction[0] == 0:
  print("Genuine Article")
else:
  print("Fake news")

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 358 stored elements and shape (1, 110429)>
  Coords	Values
  (0, 43806)	0.009864415007449725
  (0, 86453)	0.009049100869275734
  (0, 48287)	0.021196684944205953
  (0, 21055)	0.01952097566019116
  (0, 107657)	0.00915183091948528
  (0, 48100)	0.014824538274465289
  (0, 55790)	0.027781837908157643
  (0, 3018)	0.013809745778095116
  (0, 59910)	0.01790374058822816
  (0, 69866)	0.01899210251549806
  (0, 46624)	0.014675496965879413
  (0, 79871)	0.009738049383503429
  (0, 56685)	0.009958588829860499
  (0, 106176)	0.012423897141653728
  (0, 54711)	0.013563291835657233
  (0, 20751)	0.012378194979498344
  (0, 85227)	0.015156100197918325
  (0, 4778)	0.01512871952110541
  (0, 56168)	0.022748218230390655
  (0, 96975)	0.009685751690048955
  (0, 106976)	0.012501174345901794
  (0, 100117)	0.016104051065312684
  (0, 75685)	0.022212603786131387
  (0, 97096)	0.012429470269247645
  (0, 107620)	0.01643775458561188
  :	:
  (0, 109904)	0.0356832762